Phần A — groupby và agg

In [ ]:
# Thiết lập và nạp dữ liệu đã làm sạch
import sys
from pathlib import Path
# thêm m3/ vào danh sách nơi Python tìm module. Nhờ vậy from src.clean import ... mới chạy được từ trong notebooks/ 
sys.path.append(str(Path("..").resolve())) # biến .. thành đường dẫn tuyệt đối của m3/.

import pandas as pd
from src.clean import load_events, clean_events

RAW = Path("..") / "data" / "raw"
# đọc CSV, chọn 13 cột, rồi thêm ts, process_path, process_name và chuyển EVTX_Tactic, Channel sang category.
ev = clean_events(load_events(RAW / "evtx_data.csv"))
print(ev.shape)   # (4633, 16)

(4633, 16)


In [ ]:
# groupby cơ bản: chia nhóm, tính, ghép lại
g = ev.groupby("EVTX_Tactic", observed=True) # vì EVTX_Tactic là category, chỉ tạo nhóm cho các giá trị thực sự xuất hiện
print(type(g).__name__, g.ngroups) # g.ngroups: số nhóm
print("------")
print(g.size()) # : đếm số dòng (bản ghi) mỗi nhó
print("------")
print(g["EVTX_FileName"].nunique()) # : trong mỗi nhóm, đếm số file khác nhau.




DataFrameGroupBy 8
------
EVTX_Tactic
Command and Control      432
Credential Access        283
Defense Evasion          431
Discovery                167
Execution                533
Lateral Movement        1288
Persistence              411
Privilege Escalation    1088
dtype: int64
------
EVTX_Tactic
Command and Control      5
Credential Access       35
Defense Evasion         36
Discovery               10
Execution               33
Lateral Movement        47
Persistence             22
Privilege Escalation    61
Name: EVTX_FileName, dtype: int64


In [ ]:
# Nhiều chỉ số cùng lúc với named aggregation

# .agg(tên_cột_mới=("cột_nguồn", "hàm")): cú pháp named aggregation. Mỗi tham số tạo một cột kết quả, có tên do bạn đặt.
by_tactic = (ev.groupby("EVTX_Tactic", observed=True)
              .agg(n_events=("EventID", "size"), # số bản ghi.
                   n_files=("EVTX_FileName","nunique"), # số file khác nhau
                   n_eventids=("EventID", "unique"), # số loại EventID khác nhau.
                   first_seen=("ts","min")) # thời điểm sớm nhất, cho thấy agg dùng được cả với cột thời gian.
              .sort_values("n_files", ascending=False) 
            )
by_tactic["events_per_file"] = (by_tactic["n_events"] / by_tactic["n_files"]).round(1) # chia từng dòng tương ứng (vector hóa, không cần vòng lặp).
by_tactic["pct_files"] = (by_tactic["n_files"] / by_tactic["n_files"].sum() * 100 ).round(1) # Dòng này tính mỗi tactic chiếm bao nhiêu phần trăm tổng số file,
print(by_tactic)


                      n_events  n_files  \
EVTX_Tactic                               
Privilege Escalation      1088       61   
Lateral Movement          1288       47   
Defense Evasion            431       36   
Credential Access          283       35   
Execution                  533       33   
Persistence                411       22   
Discovery                  167       10   
Command and Control        432        5   

                                                             n_eventids  \
EVTX_Tactic                                                               
Privilege Escalation  [1102, 4624, 4634, 4661, 4765, 4658, 10, 1, 3,...   
Lateral Movement      [131, 168, 68, 104, 1, 4624, 5145, 18, 3, 11, ...   
Defense Evasion       [1, 7, 13, 8, 5, 104, 1102, 5156, 4663, 10, 47...   
Credential Access     [4794, 1102, 4662, 5136, 4719, 4738, 4702, 474...   
Execution             [7, 1, 11, 10, 3, 2, 13, 17, 1040, 1042, 8, 46...   
Persistence           [1102, 4662, 5136, 13,

In [ ]:
# Bảng cấp file

files = (ev.groupby("EVTX_FileName")
          .agg(tactic=("EVTX_Tactic", "first"), # trong mỗi nhóm, lấy giá trị đầu tiên của cột đó.
               n_events=("EventID", "size"),
               n_channels=("Channel","nunique"),
               has_process=("process_name", lambda s : s.notna().any()))
          .reset_index()) # sau groupby, EVTX_FileName trở thành index (nhãn dòng | nếu ko có reset index sẽ ko có cột EVTX_FileName(nếu ko nó là cột index) trong file). Lệnh này đưa nó về lại thành cột thường, để dùng làm khóa merge ở Ô 12.
print(files.shape)

n_tac = ev.groupby("EVTX_FileName")["EVTX_Tactic"].nunique()
assert n_tac.max() == 1, "Có file mang nhều Tatic" # assert điều_kiện, "thông báo khi sai"
files.sort_values("n_events", ascending=False).head(3)


(249, 5)


,EVTX_FileName,tactic,n_events,n_channels,has_process
42,LM_5145_Remote_FileCopy.evtx,Lateral Movement,869,1,False
110,Sysmon_UACME_64.evtx,Privilege Escalation,530,1,True
193,rogue_msi_url_1040_1042.evtx,Execution,351,1,False


Phần B — Đếm nhanh với value_counts và groupby nhiều cột

In [ ]:
# Đếm EventID
print(ev["EventID"].value_counts().head(5)) # groupby(cột).size() rồi sắp giảm dần

ev["EventID"].value_counts(normalize=True).head(5).mul(100).round(1)


EventID
5145    950
1       843
7       613
11      185
1040    178
Name: count, dtype: int64


EventID
5145    20.5
1       18.2
7       13.2
11       4.0
1040     3.8
Name: proportion, dtype: float64

In [ ]:
# Đếm theo cặp (Channel, EventID) và vai trò của observed
pair =(ev.groupby(["Channel","EventID"], observed=True)
        .size()
        .sort_values(ascending=False))
print(type(pair.index).__name__) # ndex có 2 tần
print(pair.head(5))
print("---------")
pair_df = pair.reset_index(name="n") # đưa hai tầng index về thành 2 cột thường, còn cột đếm đặt tên n
print(pair_df.tail(10))
print("---------")
print(ev.groupby(["Channel","EventID"], observed=False).size().shape)
# observed=False: tạo mọi tổ hợp giữa 12 kênh và 95 EventID, tức 12 × 95 = 1.140 nhóm, phần lớn có số đếm bằng 0
print(ev.groupby(["Channel","EventID"], observed=True).size().shape)
# observed=True chỉ giữ 100 tổ hợp có thật
# Với cột category, luôn dùng observed=True để bảng không phình ra.

# Sau khi astype("category"), mỗi cột nhớ danh sách của nó:
# Channel:  ['Security', 'Sysmon']
# EventID:  [1, 4624, 4625]
# observed=False ghép mọi Channel với mọi EventID: 2 × 3 = 6 nhóm.

# trong khi bảng thật thì có 3 nhóm 
# Channel	EventID
# Security	4624
# Security	4624
# Security	4625
# Sysmon	1
# Sysmon	1


MultiIndex
Channel                               EventID
Security                              5145       950
Microsoft-Windows-Sysmon/Operational  1          841
                                      7          613
                                      11         185
Application                           1040       178
dtype: int64
---------
                                 Channel  EventID  n
90  Microsoft-Windows-Sysmon/Operational       15  1
91                              Security     4625  1
92                              Security     4656  1
93                              Security     4765  1
94                              Security     4703  1
95                              Security     4724  1
96                              Security     5142  1
97                              Security     4799  1
98                              Security     4794  1
99                                System     5805  1
---------
(1140,)
(100,)


In [ ]:
# Top tiến trình theo hai cách đếm
by_rec = ev["process_name"].value_counts().head(10) # value_counts tự bỏ qua giá trị thiếu.
# print(by_rec)

by_file = (ev.dropna(subset=["process_name"])
            .groupby("process_name")["EVTX_FileName"].nunique() 
            .sort_values(ascending=False)
            .head(10)
          )
cmp = pd.DataFrame({"by_records": by_rec.index.tolist(), # .index.tolist(): lấy tên tiến trình (index), không lấy số đếm.
                    "by_files" : by_file.index.tolist()},
                  index=range(1 ,11))

print(cmp)

print("----")

# phép hiệu tập hợp, ra những phần tử có trong a mà không có trong b. sorted sắp theo bảng chữ cái.
print(sorted(set(by_rec.index) - set(by_file.index)))   # ['dllhost.exe', 'ieinstal.exe', 'mmc.exe']
print(sorted(set(by_file.index) - set(by_rec.index)))   # ['powershell.exe', 'uacme.exe', 'whoami.exe']


      by_records        by_files
1    svchost.exe         cmd.exe
2    consent.exe     svchost.exe
3        cmd.exe    explorer.exe
4         system          system
5      lsass.exe     consent.exe
6    dllhost.exe  powershell.exe
7   explorer.exe    rundll32.exe
8        mmc.exe      whoami.exe
9   rundll32.exe       lsass.exe
10  ieinstal.exe       uacme.exe
----
['dllhost.exe', 'ieinstal.exe', 'mmc.exe']
['powershell.exe', 'uacme.exe', 'whoami.exe']


In [ ]:
# Tiến trình “ồn” và tiến trình “phổ biến”

proc_stats = (ev.dropna(subset=["process_name"])
                .groupby("process_name")
                .agg(n_events=("EventID", "size"),
                     n_files=("EVTX_FileName", "nunique"))
                .sort_values("n_files", ascending=False))
proc_stats["events_per_file"] = (proc_stats["n_events"] / proc_stats["n_files"]).round(1)

proc_stats.loc[["consent.exe", "cmd.exe"]]

Phần C — Bảng chéo

In [ ]:
# crosstab: đếm tần suất hai chiều
top5 = ev["EventID"].value_counts().head(5).index
print(top5)
print("-------")
# pd.crosstab(dòng, cột) : bảng đếm hai chiều, mỗi tactic một dòng, mỗi EventID một cột, mỗi ô là số bản ghi.
ct = pd.crosstab(ev["EVTX_Tactic"], ev["EventID"])
print(ct.shape) # (8, 95)
print(ct[top5]) # chọn 5 cột cần xem trong 95 cột.
print("------")
ctn = pd.crosstab(ev["EVTX_Tactic"], ev["EventID"], normalize="index") # normalize="index": chia mỗi ô cho tổng của dòng đó, nên mỗi dòng cộng lại bằng 100%.
ctn[top5].mul(100).round(1)

Index([5145, 1, 7, 11, 1040], dtype='int64', name='EventID')
-------
(8, 95)
EventID               5145  1     7     11    1040
EVTX_Tactic                                       
Command and Control      0    13     0     0     0
Credential Access        3    60    16    13     0
Defense Evasion          2    72    51    12     0
Discovery               15     0     0     0     0
Execution                0   101     8    33   178
Lateral Movement       929    61     7    10     0
Persistence              0   235     4    28     0
Privilege Escalation     1   301   527    89     0
------


EventID,5145,1,7,11,1040
EVTX_Tactic,,,,,
Command and Control,0.0,3.0,0.0,0.0,0.0
Credential Access,1.1,21.2,5.7,4.6,0.0
Defense Evasion,0.5,16.7,11.8,2.8,0.0
Discovery,9.0,0.0,0.0,0.0,0.0
Execution,0.0,18.9,1.5,6.2,33.4
Lateral Movement,72.1,4.7,0.5,0.8,0.0
Persistence,0.0,57.2,1.0,6.8,0.0
Privilege Escalation,0.1,27.7,48.4,8.2,0.0


In [ ]:
# pivot_table: đếm theo file thay vì theo bản ghi

# giống crosstab, nhưng cho chọn cột giá trị (values) và hàm tổng hợp (aggfunc).
# values="EVTX_FileName", aggfunc="nunique": mỗi ô là số file có chứa EventID đó, không phải số bản ghi.
pt = ev.pivot_table(index="EVTX_Tactic", columns="EventID",
                    values="EVTX_FileName", aggfunc="nunique",
                    fill_value=0, observed=True) # fill_value=0: tổ hợp không có dữ liệu thì điền 0 thay vì NaN.
print(pt[top5])
print("----------")
(ev.loc[ev["EventID"] == 5145]
   .groupby("EVTX_FileName").size()
   .sort_values(ascending=False)
   .head(3))

EventID               5145  1     7     11    1040
EVTX_Tactic                                       
Command and Control      0     1     0     0     0
Credential Access        2     8     4     7     0
Defense Evasion          1    17     9     5     0
Discovery                3     0     0     0     0
Execution                0    29     7     8     1
Lateral Movement         7    19     4     4     0
Persistence              0    10     2     3     0
Privilege Escalation     1    49    14    25     0
----------


EVTX_FileName
LM_5145_Remote_FileCopy.evtx      869
LM_REMCOM_5145_TargetHost.evtx     30
LM_renamed_psexecsvc_5145.evtx     22
dtype: int64

In [ ]:
# Rút gọn tên kênh rồi pivot

# .astype(str): Channel đang là category; chuyển về chuỗi để dùng các hàm .str.
# .str.split("/").str[0]: tách tại / rồi lấy phần đầu. Ví dụ "Microsoft-Windows-Sysmon/Operational" thành "Microsoft-Windows-Sysmon".
# .str.split("-").str[-1]: tách tại - rồi lấy phần cuối ([-1]), ra "Sysmon". Kênh "Security" không có / hay - nên giữ nguyên.
short = ev["Channel"].astype(str).str.split("/").str[0].str.split("-").str[-1]
print(short.value_counts().head(4))   # Sysmon 2401, Security 1496, Application 393, ...

# ev.assign(channel_short=short): tạo bản sao của ev có thêm cột mới, không sửa ev gốc. Tiện khi cần cột tạm cho một phép tính.
pt2 = (ev.assign(channel_short=short)
         .pivot_table(index="EVTX_Tactic", columns="channel_short",
                      values="EVTX_FileName", aggfunc="nunique",
                      fill_value=0, observed=True))
pt2[["Sysmon", "Security", "Application", "System"]]

Channel
Sysmon                     2401
Security                   1496
Application                 393
RemoteConnectionManager     228
Name: count, dtype: int64


channel_short,Sysmon,Security,Application,System
EVTX_Tactic,,,,
Command and Control,2,2,0,0
Credential Access,19,11,3,1
Defense Evasion,28,5,1,2
Discovery,6,4,0,0
Execution,29,1,1,0
Lateral Movement,28,12,2,2
Persistence,16,3,0,0
Privilege Escalation,51,9,0,1


Phần D — merge với bảng mapping ATT&CK

In [46]:
# Gộp mã tactic vào bảng file

# files = (ev.groupby("EVTX_FileName")
#           .agg(tactic=("EVTX_Tactic", "first"), # trong mỗi nhóm, lấy giá trị đầu tiên của cột đó.
#                n_events=("EventID", "size"),
#                n_channels=("Channel","nunique"),
#                has_process=("process_name", lambda s : s.notna().any()))
#           .reset_index())

# bảng tra cứu nhỏ, mỗi tactic một dòng.
# tactic_id là mã chính thức trên trang MITRE ATT&CK
# kill_chain_order là vị trí cột của tactic trong ma trận ATT&CK Enterprise (Execution là cột 4, C2 là cột 12).
tactics = pd.DataFrame({
    "tactic": ["Execution", "Persistence", "Privilege Escalation", "Defense Evasion",
               "Credential Access", "Discovery", "Lateral Movement", "Command and Control"],
    "tactic_id": ["TA0002", "TA0003", "TA0004", "TA0005",
                  "TA0006", "TA0007", "TA0008", "TA0011"],
    "kill_chain_order": [4, 5, 6, 7, 8, 9, 10, 12],
})

# files.merge(tactics, on="tactic"): với mỗi dòng của files, tìm dòng trong tactics có cùng giá trị tactic rồi ghép các cột lại. Tương tự JOIN ... ON trong SQL.
# how="left": giữ mọi dòng của bảng trái (files). Dòng nào không khớp vẫn còn, chỉ có cột bên phải là NaN
# validate="many_to_one": khẳng định nhiều file ứng với một tactic, và bên phải không có khóa trùng. Sai thì báo lỗi ngay
# indicator=True: thêm cột _merge, ghi mỗi dòng là both (khớp), left_only hay right_only.
files["tactic"] = files["tactic"].astype(str) # cột này đang là category (lấy từ EVTX_Tactic), còn khóa bên tactics là str. Đưa hai bên về cùng kiểu để merge khớp chắc chắn.
files_m = files.merge(tactics, on="tactic", how="left", validate="many_to_one", indicator=True)
print(files_m.shape)
print(files_m["_merge"].value_counts().to_dict())

# groupby([... 3 cột ...]) rồi sắp theo kill_chain_order: bảng số mẫu được sắp theo thứ tự các giai đoạn tấn công, dễ đọc hơn sắp theo bảng chữ cái.
(files_m.groupby(["kill_chain_order", "tactic_id", "tactic"])
        .size()
        .reset_index(name="n_files"))


(249, 8)
{'both': 249, 'left_only': 0, 'right_only': 0}


,kill_chain_order,tactic_id,tactic,n_files
0,4,TA0002,Execution,33
1,5,TA0003,Persistence,22
2,6,TA0004,Privilege Escalation,61
3,7,TA0005,Defense Evasion,36
4,8,TA0006,Credential Access,35
5,9,TA0007,Discovery,10
6,10,TA0008,Lateral Movement,47
7,12,TA0011,Command and Control,5


In [50]:
# Ba cái bẫy của merge

# Bẫy 1: bảng mapping thiếu khóa
missing = tactics[tactics["tactic"] != "Discovery"]
print(files.merge(missing, on="tactic", how="left", indicator=True)["_merge"].value_counts().to_dict())
# {'both': 239, 'left_only': 10, 'right_only': 0}
print(files.merge(missing, on="tactic", how="inner").shape)   # (239, 7)  <- mất 10 file, không báo gì

# Bẫy 2: bảng mapping có khóa trùng
dup = pd.concat([tactics, tactics.iloc[[0]]], ignore_index=True)
try:
    files.merge(dup, on="tactic", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e)[:70])
# MergeError: Merge keys are not unique in right dataset; not a many-to-one merge
print(files.merge(dup, on="tactic", how="left").shape)   # (282, 7)  <- 249 thành 282 dòng

# Bẫy 3: khoảng trắng thừa trong khóa
messy = tactics.copy()
messy.loc[0, "tactic"] = "Execution "
print(files.merge(messy, on="tactic", how="left", indicator=True)["_merge"].value_counts().to_dict())
# {'both': 216, 'left_only': 33, ...}
messy["tactic"] = messy["tactic"].str.strip()
print(files.merge(messy, on="tactic", how="left", indicator=True)["_merge"].value_counts().to_dict())
# {'both': 249, 'left_only': 0, ...}

{'both': 239, 'left_only': 10, 'right_only': 0}
(239, 7)
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

D
(282, 7)
{'both': 216, 'left_only': 33, 'right_only': 0}
{'both': 249, 'left_only': 0, 'right_only': 0}


Phần E — concat: nối hai nguồn log

In [55]:
# Ghép EVTX-ATTACK-SAMPLES với OTRF theo một schema chung
a = ev.assign(source="evtx_attack_samples")[["source", "Channel", "EventID", "process_name", "ts"]]
# print(a)

otrf = pd.read_json(RAW / "empire_launcher_vbs.zip", lines=True, compression="zip")
# print(otrf)
# : dựng bảng OTRF với đúng tên cột như bảng a. Đây là bước quan trọng nhất trước khi concat: hai nguồn đặt tên khác nhau phải được đổi về một schema chung.
b = pd.DataFrame({
    "source": "otrf_empire_launcher_vbs",
    "Channel": otrf["Channel"],
    "EventID": otrf["EventID"],
    "process_name": otrf["Image"].str.extract(r"([^\\/]+)$", expand=False).str.lower(),
    "ts": pd.to_datetime(otrf["@timestamp"], utc=True),
})

# print(b)

both = pd.concat([a, b], ignore_index=True) # ignore_index=True đánh số lại index từ 0 đến 6699, tránh hai dòng cùng mang nhãn 0.
print(a.shape, b.shape, both.shape) 

both.groupby("source").agg(
    n=("EventID", "size"),
    n_eventids=("EventID", "nunique"),
    pct_proc=("process_name", lambda s: round(s.notna().mean() * 100, 1)),
)


(4633, 5) (2067, 5) (6700, 5)


,n,n_eventids,pct_proc
source,,,
evtx_attack_samples,4633,95,53.1
otrf_empire_launcher_vbs,2067,37,41.9
